In [0]:
# Read from the Bronze Delta Lake table
df_countries = (
    spark.read
    .format("delta")
    .load("abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_countries")
)


In [0]:
display(df_countries)

In [0]:
import pyspark.sql.functions as F

# 1. Check Row Counts
raw_df_countries = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("delimiter", ",")
    .csv("abfss://raw@mynetflixproject.dfs.core.windows.net/netflix_countries/")
)
raw_count = raw_df_countries.count()
bronze_count = df_countries.count()
count_difference = bronze_count - raw_count

# 2. Standardise schemas to strings for an exact text content comparison
raw_standardised = raw_df_countries.select([F.col(c).cast("string").alias(c) for c in raw_df_countries.columns])
bronze_standardised = df_countries.select([F.col(c).cast("string").alias(c) for c in raw_df_countries.columns])

# 3. Check for specific content differences
rows_in_raw_not_bronze = raw_standardised.subtract(bronze_standardised).count()
rows_in_bronze_not_raw = bronze_standardised.subtract(raw_standardised).count()

# 4. Print Complete Validation Summary
print("=" * 45)
print("🔍 COMPLETE RAW VS. BRONZE VERIFICATION")
print("=" * 45)
print(f"Raw Row Count:                {raw_count:,}")
print(f"Bronze Row Count:             {bronze_count:,}")
print(f"Row Count Difference:         {count_difference:,}")
print("-" * 45)
print(f"Rows in Raw but NOT Bronze:   {rows_in_raw_not_bronze}")
print(f"Rows in Bronze but NOT Raw:   {rows_in_bronze_not_raw}")
print("=" * 45)

# Final Verdict Check
counts_match = (raw_count == bronze_count)
content_matches = (rows_in_raw_not_bronze == 0 and rows_in_bronze_not_raw == 0)

if counts_match and content_matches:
    print("✅ SUCCESS: Data is 100% identical in both count and content!")
else:
    print("❌ ERROR: Mismatch detected! Check the numbers above.")
print("=" * 45)


In [0]:
from pyspark.sql import functions as F

df_countries = spark.read.format("delta").load(
    "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_countries"
)

country_col = F.col("country")
country_trimmed = F.trim(country_col)

# 1. Single-pass summary of quality issues in 'country'
df_countries.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(country_trimmed).alias("distinct_countries"),
    F.sum(F.when(country_col.isNull() | (country_trimmed == ""), 1).otherwise(0)).alias("dq_null_or_blank"),
    F.sum(F.when(country_col != country_trimmed, 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(country_col.rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    F.sum(F.when(country_col.contains(","), 1).otherwise(0)).alias("dq_contains_comma"),
    F.sum(F.when(country_trimmed.rlike(r"^[0-9\W_]+$"), 1).otherwise(0)).alias("dq_numeric_or_symbol_only")
).display()

# 2. Print the full distinct list of countries to spot naming variants or typos
(
    df_countries
    .groupBy(country_trimmed.alias("country"))
    .count()
    .orderBy("country")
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Strip commas/whitespace, collapse repeated spaces, and apply smart Title Case
base_country = F.regexp_replace(
    F.regexp_replace(
        F.initcap(
            F.trim(
                F.regexp_replace(
                    F.regexp_replace(F.col("country"), r"^[,\s]+|[,\s]+$", ""),
                    r"\s+",
                    " "
                )
            )
        ),
        r"\bOf\b", "of"
    ),
    r"\bAnd\b", "and"
)

# 2. Map common abbreviations / short forms to canonical full country names
country_aliases = {
    "Usa": "United States",
    "Us": "United States",
    "U.s.": "United States",
    "U.s.a.": "United States",
    "United States of America": "United States",
    "Uk": "United Kingdom",
    "U.k.": "United Kingdom",
    "Great Britain": "United Kingdom",
    "England": "United Kingdom",
    "Uae": "United Arab Emirates",
    "U.a.e.": "United Arab Emirates"
}

mapping_expr = F.create_map([F.lit(x) for item in country_aliases.items() for x in item])

staged_countries_df = df_countries.withColumn(
    "country",
    F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), F.lit(None).cast("string"))
     .otherwise(F.coalesce(mapping_expr[base_country], base_country))
)

# 3. Quarantine invalid country values (NULL, blank, or pure numbers/symbols)
is_invalid_country = (
    F.col("country").isNull() |
    F.col("country").rlike(r"^[0-9\W_]+$")
)

quarantine_country_col_df = staged_countries_df.filter(is_invalid_country)
clean_countries_df = staged_countries_df.filter(~is_invalid_country)

# 4. Validate the cleaned country column
clean_countries_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("country").alias("distinct_countries"),
    F.sum(F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), 1).otherwise(0)).alias("val_null_or_blank"),
    F.sum(F.when(F.col("country").contains(","), 1).otherwise(0)).alias("val_remaining_commas"),
    F.sum(F.when(F.col("country").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces")
).display()


In [0]:
from pyspark.sql import functions as F

# 1. Automated quality gate checks (all error columns must equal 0)
clean_countries_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("country").alias("distinct_countries"),
    F.sum(F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), 1).otherwise(0)).alias("val_null_or_blank"),
    F.sum(F.when(F.col("country").contains(","), 1).otherwise(0)).alias("val_remaining_commas"),
    F.sum(F.when(F.col("country").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces"),
    F.sum(F.when(F.col("country").isin("Usa", "Us", "Uk", "Uae", "U.s.", "U.k."), 1).otherwise(0)).alias("val_unmapped_short_forms")
).display()

# 2. Verify United Kingdom (602) and United States (2610) merged cleanly without trailing commas
(
    clean_countries_df
    .filter(F.col("country").isin("United Kingdom", "United States", "United Arab Emirates"))
    .groupBy("country")
    .count()
    .orderBy("country")
    .display()
)


xx


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load valid parent show_ids from Silver netflix_titles
silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

id_str = F.trim(F.col("show_id").cast("string"))

# 1. Single-pass audit of show_id quality AND composite duplicate (show_id, country) pairs
clean_countries_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(id_str.isNull() | (id_str == ""), 1).otherwise(0)).alias("dq_null_show_id"),
    F.sum(F.when(id_str.isNotNull() & ~id_str.rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("dq_non_digit_show_id"),
    F.countDistinct(id_str).alias("distinct_show_ids"),
    (F.count("*") - F.countDistinct(id_str, F.lower(F.col("country")))).alias("dq_duplicate_pairs")
).display()

# 2. Inspect any duplicate (show_id, country) pairs side-by-side (e.g., if a show had both "United States" and "United States,")
w_pair = Window.partitionBy(id_str, F.lower(F.col("country")))

(
    clean_countries_df
    .withColumn("pair_cnt", F.count("*").over(w_pair))
    .filter(F.col("pair_cnt") > 1)
    .orderBy("show_id", "country")
    .display()
)

# 3. Check for orphan show_ids missing from Silver netflix_titles
orphan_countries_df = (
    clean_countries_df
    .withColumn("show_id_clean", id_str)
    .join(silver_titles_ids, F.col("show_id_clean") == silver_titles_ids.show_id, "left_anti")
)

print(f"Orphan show_id rows in netflix_countries: {orphan_countries_df.count()}")
orphan_countries_df.display()


In [0]:
from pyspark.sql import functions as F

silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

# 1. Trim whitespace and cast show_id cleanly to STRING
staged_id_df = clean_countries_df.withColumn("show_id", F.trim(F.col("show_id").cast("string")))

# 2. Check foreign-key match against Silver netflix_titles
staged_with_fk = (
    staged_id_df.alias("c")
    .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
    .select(
        F.col("c.*"),
        F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
    )
)

# 3. Quarantine rows with missing, non-digit, or orphan show_id
is_invalid_show_id = (
    F.col("show_id").isNull() |
    (F.col("show_id") == "") |
    ~F.col("show_id").rlike(r"^[0-9]+$") |
    F.col("_is_orphan")
)

quarantine_countries_id_df = (
    staged_with_fk
    .filter(is_invalid_show_id)
    .withColumn(
        "quarantine_reason",
        F.when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
         .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
         .otherwise("Unknown show_id error")
    )
    .drop("_is_orphan")
)

# 4. Keep valid show_id rows as STRING and deduplicate composite key (show_id, country)
clean_countries_df = (
    staged_with_fk
    .filter(~is_invalid_show_id)
    .drop("_is_orphan")
    .dropDuplicates(["show_id", "country"])
)

# 5. Validate show_id
clean_countries_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("show_id").alias("distinct_show_ids"),
    F.countDistinct("show_id", "country").alias("distinct_pairs"),
    F.sum(F.when(F.col("show_id").isNull() | ~F.col("show_id").rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("val_invalid_show_ids")
).display()

print(f"Quarantined orphan/invalid show_id rows: {quarantine_countries_id_df.count()}")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_countries"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_countries"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_countries"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_countries"

def process_countries_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Strip commas/spaces, apply smart Title Case, and map short forms
    base_country = F.regexp_replace(
        F.regexp_replace(
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.col("country"), r"^[,\s]+|[,\s]+$", ""),
                        r"\s+",
                        " "
                    )
                )
            ),
            r"\bOf\b", "of"
        ),
        r"\bAnd\b", "and"
    )

    country_aliases = {
        "Usa": "United States",
        "Us": "United States",
        "U.s.": "United States",
        "U.s.a.": "United States",
        "United States of America": "United States",
        "Uk": "United Kingdom",
        "U.k.": "United Kingdom",
        "Great Britain": "United Kingdom",
        "England": "United Kingdom",
        "Uae": "United Arab Emirates",
        "U.a.e.": "United Arab Emirates"
    }
    mapping_expr = F.create_map([F.lit(x) for item in country_aliases.items() for x in item])

    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "country",
            F.when(F.col("country").isNull() | (F.trim(F.col("country")) == ""), F.lit(None).cast("string"))
             .otherwise(F.coalesce(mapping_expr[base_country], base_country))
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("c")
        .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("c.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Unified quarantine rules
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("country").isNull() | F.col("country").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("country").isNull() | F.col("country").rlike(r"^[0-9\W_]+$"), "Invalid or Missing country")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk.filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "country"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.country = source.country"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_countries_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_countries.")
